# NN 世界模型實驗 — Colab 啟動器

一鍵在 Google Colab 跑本 repo 的 25 個實驗。

- **A 層 MVP（25 個）**：純 numpy、零依賴、CPU 即可，隔離每篇論文的核心機制。
- **B 層真實資料（11 個）**：`*_real.py`，自動下載公開資料。
- **C 層（規劃最大實驗）**：需 GPU 時，Colab 的 T4(16GB) 比本機 1080 Ti(11GB) 更寬裕。

用法：`執行階段 → 全部執行`。要 GPU 時先 `執行階段 → 變更執行階段類型 → T4 GPU`。

> 對應 GitHub：https://github.com/a0935951152-droid/Allen-Data-Science

## 1. 取得程式碼 + 安裝依賴

In [ ]:
!git clone --depth 1 https://github.com/a0935951152-droid/Allen-Data-Science.git repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install numpy
print('\n✅ repo 就緒')

## 2. 檢查執行環境（GPU 可選）
MVP 不需要 GPU；C 層要 GPU 時，這裡確認 T4 是否已掛上。

In [ ]:
import subprocess, sys
print('Python', sys.version.split()[0])
g = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', g.stdout.strip() if g.returncode == 0 else '（目前為 CPU，MVP 足夠；C 層再開 GPU）')

## 3. 一鍵跑全部 25 個 A 層 MVP
每支印出可驗證結果（還原的方程、估計的指數、方差喇叭口、群純度…）。

In [ ]:
import glob, subprocess, os
mvps = sorted(glob.glob('experiments/*/*_mvp.py')
              + glob.glob('experiments/*/sindy_lorenz.py')
              + glob.glob('experiments/*/lorenz_butterfly.py'))
for f in mvps:
    print('='*70); print('▶', f)
    r = subprocess.run(['python', f], capture_output=True, text=True, timeout=180)
    print(r.stdout.strip() or r.stderr[-400:])
print('\n✅ 全部 MVP 跑完，共', len(mvps), '支')

## 4. 跑 11 個 B 層真實資料實驗（自動下載公開資料）
SILSO 黑子、Mauna Loa CO₂、JHU COVID、JPL 星曆、Gaia 恆星… 首次執行會連網下載。

In [ ]:
import glob, subprocess
for f in sorted(glob.glob('experiments/*/*_real.py')):
    print('='*70); print('▶', f)
    r = subprocess.run(['python', f], capture_output=True, text=True, timeout=300)
    print(r.stdout.strip() or r.stderr[-400:])

## 5. 跑單一實驗
把路徑換成任一實驗即可。

In [ ]:
!python experiments/p3-2-score-particles/score_mvp.py

## 6. C 層：用 GPU 擴展（範本）

C 層「規劃最大實驗」多數需要 torch/GPU。步驟：
1. 上方 `變更執行階段類型 → T4 GPU`，重跑 cell 1–2。
2. `!pip -q install torch pyro-ppl` 安裝該實驗 README「③ 規劃最大實驗」所列套件。
3. 依該實驗資料夾 README 的設定/度量/判準實作。

下面是一個 GPU 冒煙測試，確認 torch 能用 GPU：

In [ ]:
try:
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    x = torch.randn(2000, 2000, device=dev)
    y = (x @ x).sum().item()
    print(f'torch {torch.__version__} on {dev}: matmul ok ({y:.1f})')
except ImportError:
    print('尚未安裝 torch；C 層需要時執行  !pip install torch')